# Notebook 02 - Yelp Data Preprocessing

This notebook specifically focuses on the data preprocessing code to help us filter our datasets before they are split for downstream use in training and evaluation. This part of the pipeline also includes our pick for k-core, filtering by open restaurants, features for our towers and feature-engineered choices, and splitting before save. 

### Filtering of Business Data

In [25]:
import pandas as pd
import json

yelp_business_data_path = "../Yelp JSON/yelp_dataset/yelp_academic_dataset_business.json"
with open(yelp_business_data_path, "r") as f:
    yelp_business_df = pd.DataFrame(json.loads(line) for line in f)

# Filter to open restaurants/food businesses only
mask = (
    yelp_business_df["categories"].str.contains("Restaurants|Food", na=False) &
    (yelp_business_df["is_open"] == 1)
)
yelp_open_restaurants_df = yelp_business_df[mask].copy()

print(f"{len(yelp_open_restaurants_df):} open restaurant/food businesses")

44582 open restaurant/food businesses


### Filtration of Reviews by Business ID

In [26]:
yelp_review_data_path = "../Yelp JSON/yelp_dataset/yelp_academic_dataset_review.json"
with open(yelp_review_data_path, "r") as f:
    yelp_reviews_df = pd.DataFrame(json.loads(line) for line in f)

# Filter reviews to open restaurants only
open_restaurant_ids = set(yelp_open_restaurants_df["business_id"])
yelp_reviews_df = yelp_reviews_df[yelp_reviews_df["business_id"].isin(open_restaurant_ids)].copy()

print(f"{len(yelp_reviews_df):} reviews for open restaurants")

4104995 reviews for open restaurants


### K-Core Filtering

From the EDA notebook, we got the following table of values for the k-core statistics.  

| k-value | # of users | # of reviews | avg # of reviews/user|
| --- | --- | --- | --- |
| k = 5 | 210,133 users | 3,149,767 reviews | 15.0 avg reviews/user |
| k = 6 | 166,691 users | 2,932,557 reviews | 17.6 avg reviews/user |
| k = 7 | 136,534 users | 2,751,615 reviews | 20.2 avg reviews/user |

Based on the data, we decided to pick k = 5 as our k-core value. The reason behind picking k = 5  
was because we wanted to have a balanced tradeoff between the number of users, number of reviews,  
and the roughly average number of reviews/user.   

With a minimum baseline of 5 reviews for each user, this would also help combat reviews from low-activity users  
from affecting our recommendations significantly, ensuring that the quality of recommendations is up to a higher standard.

We are applying iterative k-core filtering to ensure every user has at least 5 reviews AND every business has at least 5 reviews simultaneously. Although the business JSON had a minimum of 5 reviews, after filtering to open restaurants only, some businesses lost reviews and dropped below k = 5. As previously mentioned in EDA, we expect to see slightly lower numbers than what the table states. 

In [27]:
k = 5
while True:
    user_counts = yelp_reviews_df["user_id"].value_counts()
    business_counts = yelp_reviews_df["business_id"].value_counts()

    val_users = set(user_counts[user_counts >= k].index)
    val_businesses = set(business_counts[business_counts >= k].index)

    k_filter_mask = yelp_reviews_df[
        yelp_reviews_df["user_id"].isin(val_users) &
        yelp_reviews_df["business_id"].isin(val_businesses)
    ]

    if len(k_filter_mask) == len(yelp_reviews_df):
        break

    yelp_reviews_df = k_filter_mask.copy()

print(f"{yelp_reviews_df['user_id'].nunique():} users, {yelp_reviews_df['business_id'].nunique():} businesses, {len(yelp_reviews_df):} reviews after k-core filtering")

166488 users, 39140 businesses, 2323580 reviews after k-core filtering


As evidenced in the results above, the number of users, businesses, and reviews in consideration reduced.  
However, roughly 166K users, 39K businesses, and 2.3M reviews is sufficient for creating meaningful training, validation, and test splits. 

### Filtering of Users (Matching with Reviews User IDs)

In [28]:
yelp_users_data_path = "../Yelp JSON/yelp_dataset/yelp_academic_dataset_user.json"
with open(yelp_users_data_path, "r") as f:
    yelp_users_df = pd.DataFrame(json.loads(line) for line in f)

# Filter to only users present in k-core filtered reviews
review_user_ids = set(yelp_reviews_df["user_id"])
yelp_users_df = yelp_users_df[yelp_users_df["user_id"].isin(review_user_ids)].copy()

# Checking if the user count matches
assert len(yelp_users_df) == yelp_reviews_df["user_id"].nunique(), "User count mismatch!"
print(f"User count matches: {len(yelp_users_df):} users")

User count matches: 166488 users


### Feature Engineering

We added filtering logic above to handle the filtering of the three key datasets for our systems. Now, we'll work on the feature engineering to curate the features that will be helpful for generating proper recommendations. 

#### User Tower - feature 1: avg_stars_given

In [30]:
# `avg_stars_given` is derived directly from the `average_stars` field in the user dataset. 
# It captures the user's general rating tendency, which helps the model distinguish between a 4-star rating from a harsh rater vs. a lenient one.

yelp_users_df["avg_stars_given"] = yelp_users_df["average_stars"]

print(yelp_users_df["avg_stars_given"].describe())
print(f"\nSample values:\n{yelp_users_df[['user_id', 'avg_stars_given']].head(10).to_string(index=False)}")

count    166488.000000
mean          3.817245
std           0.649999
min           1.000000
25%           3.480000
50%           3.900000
75%           4.260000
max           5.000000
Name: avg_stars_given, dtype: float64

Sample values:
               user_id  avg_stars_given
qVc8ODYU5SZjKXVBgXdI7w             3.91
j14WgRoU_-2ZE1aw1dXrJg             3.74
2WnXYQFK0hXEoTxPtV2zvg             3.32
SZDeASXq7o05mMNLshsdIA             4.27
hA5lMy-EnncsH4JoR-hFGQ             3.54
q_QQ5kBBwlCcbL1s4NVK3g             3.85
cxuxXkcihfCbqt5Byrup8Q             2.75
AUi8MPWJ0mLkMfwbui27lg             3.40
SgiBkhXeqIKl1PlFpZOycQ             3.75
QF1Kuhs8iwLWANNZxebTow             4.11


#### User Tower - feature 2: review_count_log

In [31]:
# `review_count` from the user dataset measures how active a user is on Yelp. 
# However, review counts are heavily right-skewed as most users have only a handful of reviews while a small number have hundreds or thousands. 
# As a result, we've applied a log1p transform to make the feature more well-behaved for the model. 

import numpy as np

yelp_users_df["review_count_log"] = np.log1p(yelp_users_df["review_count"])

print(yelp_users_df[["review_count", "review_count_log"]].describe())
print(f"\nSample values:\n{yelp_users_df[['user_id', 'review_count', 'review_count_log']].head(10).to_string(index=False)}")

        review_count  review_count_log
count  166488.000000     166488.000000
mean       89.240930          3.637606
std       211.590374          1.162186
min         1.000000          0.693147
25%        15.000000          2.772589
50%        28.000000          3.367296
75%        76.000000          4.343805
max     17473.000000          9.768469

Sample values:
               user_id  review_count  review_count_log
qVc8ODYU5SZjKXVBgXdI7w           585          6.373320
j14WgRoU_-2ZE1aw1dXrJg          4333          8.374246
2WnXYQFK0hXEoTxPtV2zvg           665          6.501290
SZDeASXq7o05mMNLshsdIA           224          5.416100
hA5lMy-EnncsH4JoR-hFGQ            79          4.382027
q_QQ5kBBwlCcbL1s4NVK3g          1221          7.108244
cxuxXkcihfCbqt5Byrup8Q            12          2.564949
AUi8MPWJ0mLkMfwbui27lg           109          4.700480
SgiBkhXeqIKl1PlFpZOycQ           682          6.526495
QF1Kuhs8iwLWANNZxebTow           607          6.410175


#### User Tower - feature 3: yelping_since_years

In [34]:
# `yelping_since` is a date string that represents when the user joined Yelp. 

reference_date = pd.to_datetime(yelp_users_df["yelping_since"]).max()

yelp_users_df["yelping_since_years"] = (
    (reference_date - pd.to_datetime(yelp_users_df["yelping_since"])).dt.days / 365.25
)

print(f"Reference date: {reference_date}")
print(yelp_users_df["yelping_since_years"].describe())
print(f"\nSample values:\n{yelp_users_df[['user_id', 'yelping_since', 'yelping_since_years']].head(10).to_string(index=False)}")

Reference date: 2022-01-17 15:25:41
count    166488.000000
mean          8.254171
std           2.947440
min           0.000000
25%           6.220397
50%           8.325804
75%          10.417522
max          17.264887
Name: yelping_since_years, dtype: float64

Sample values:
               user_id       yelping_since  yelping_since_years
qVc8ODYU5SZjKXVBgXdI7w 2007-01-25 16:47:26            14.976044
j14WgRoU_-2ZE1aw1dXrJg 2009-01-25 04:35:42            12.977413
2WnXYQFK0hXEoTxPtV2zvg 2008-07-25 10:41:00            13.481177
SZDeASXq7o05mMNLshsdIA 2005-11-29 04:38:33            16.134155
hA5lMy-EnncsH4JoR-hFGQ 2007-01-05 19:40:59            15.030801
q_QQ5kBBwlCcbL1s4NVK3g 2005-03-14 20:26:35            16.843258
cxuxXkcihfCbqt5Byrup8Q 2009-02-24 03:09:06            12.895277
AUi8MPWJ0mLkMfwbui27lg 2010-01-07 18:32:04            12.024641
SgiBkhXeqIKl1PlFpZOycQ 2006-08-25 16:47:25            15.394935
QF1Kuhs8iwLWANNZxebTow 2009-04-27 20:25:54            12.722793


#### User Tower - feature 4: is_elite

In [35]:
# is_elite flag indicates if a user is of elite status or not. 
# elite users are recognized by Yelp for writing very high-quality reviews, 
# so can be considered more trustworthy + engaged reviewers

yelp_users_df["is_elite"] = (yelp_users_df["elite"].str.strip() != "").astype(int)

print(yelp_users_df["is_elite"].value_counts())
print(f"\nElite rate: {yelp_users_df['is_elite'].mean():.2%}")

is_elite
0    131238
1     35250
Name: count, dtype: int64

Elite rate: 21.17%
